In [0]:
# Cell 1 — Install required libraries for Notebook 02
%pip install presidio-analyzer presidio-anonymizer
%pip install spacy

dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 136.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.7/32.7 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 861.5/861.5 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 135.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 60.1 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.8
    Not uninstalling click at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-6ce5632d-00c2-48d4-b4a8-59016d5e585a
    Can't uninstall 'click'. No files were found to uninstall.
  Attempting uninstall: cffi
    Found existing installation: cffi 1.17.1
    Not uninstalling cffi at /databricks/python3/lib/python3.

In [0]:
dbutils.library.restartPython()

In [0]:
# Cell 2 — Download Synthea data directly into memory
import requests
import zipfile
import io
import pandas as pd

print("Downloading Synthea data...")
response = requests.get(
    "https://mitre.box.com/shared/static/9iglv8kbs1pfi7z8phjl9sbpjk08spze.zip"
)
print("✅ Downloaded!")

# Open zip directly from memory — no file system needed
zip_file = zipfile.ZipFile(io.BytesIO(response.content))
print("✅ Zip opened in memory!")

# List available files
print("\nFiles available:")
for name in zip_file.namelist():
    if name.endswith(".csv"):
        print(f"  📄 {name}")

✅ Downloaded!
✅ Zip opened in memory!

Files available:
  📄 10k_synthea_covid19_csv/medications.csv
  📄 10k_synthea_covid19_csv/providers.csv
  📄 10k_synthea_covid19_csv/payer_transitions.csv
  📄 10k_synthea_covid19_csv/imaging_studies.csv
  📄 10k_synthea_covid19_csv/supplies.csv
  📄 10k_synthea_covid19_csv/payers.csv
  📄 10k_synthea_covid19_csv/allergies.csv
  📄 10k_synthea_covid19_csv/procedures.csv
  📄 10k_synthea_covid19_csv/organizations.csv
  📄 10k_synthea_covid19_csv/conditions.csv
  📄 10k_synthea_covid19_csv/careplans.csv
  📄 10k_synthea_covid19_csv/encounters.csv
  📄 10k_synthea_covid19_csv/devices.csv
  📄 10k_synthea_covid19_csv/immunizations.csv
  📄 10k_synthea_covid19_csv/patients.csv
  📄 10k_synthea_covid19_csv/observations.csv


In [0]:
# Cell 3 — Load raw CSVs into Bronze Delta tables
files_to_load = ["patients", "conditions", "medications", "procedures"]

for table in files_to_load:
    print(f"Loading {table}...")
    
    # Find the file in the zip
    matching = [f for f in zip_file.namelist() if f.endswith(f"{table}.csv")]
    
    # Read CSV from memory into pandas
    pandas_df = pd.read_csv(zip_file.open(matching[0]))
    
    # Convert pandas to Spark DataFrame
    spark_df = spark.createDataFrame(pandas_df)
    
    # Save as Bronze Delta table
    spark_df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"bronze_{table}")
    
    print(f"✅ bronze_{table} — {len(pandas_df):,} rows")

print("\n🎉 All Bronze tables created!")

Loading patients...
✅ bronze_patients — 12,352 rows
Loading conditions...
✅ bronze_conditions — 114,544 rows
Loading medications...
✅ bronze_medications — 431,262 rows
Loading procedures...
✅ bronze_procedures — 100,427 rows

🎉 All Bronze tables created!


In [0]:
# Cell 4 — De-identify Bronze data → Save as Silver tables
# This is the HIPAA-aware step using Microsoft Presidio

import subprocess
import pandas as pd
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

# Download spaCy English model — Presidio needs this
print("Loading spaCy language model...")
subprocess.run(["python", "-m", "spacy", "download", "en_core_web_lg"], 
               capture_output=True)
print("✅ spaCy model ready!")

# Initialize Presidio engines
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()
print("✅ Presidio engines initialized!")

# Function to de-identify a single text value
def deidentify_text(text):
    if text is None or str(text).strip() == "":
        return text
    text = str(text)
    # Analyze — find all PII entities
    results = analyzer.analyze(
        text=text,
        entities=["PERSON", "US_SSN", "LOCATION", "DATE_TIME", "PHONE_NUMBER"],
        language="en"
    )
    # Anonymize — replace PII with labels
    if results:
        anonymized = anonymizer.anonymize(text=text, analyzer_results=results)
        return anonymized.text
    return text

# De-identify patients table — most sensitive file
print("\nDe-identifying patients table...")
patients_bronze = spark.table("bronze_patients").toPandas()

# These are the columns that contain PHI
phi_columns = ["FIRST", "LAST", "SSN", "ADDRESS", "CITY", "BIRTHDATE"]

for col in phi_columns:
    if col in patients_bronze.columns:
        patients_bronze[col] = patients_bronze[col].apply(deidentify_text)
        print(f"  ✅ {col} de-identified")

# Save as Silver Delta table
silver_patients = spark.createDataFrame(patients_bronze)
silver_patients.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_patients")

print(f"\n✅ silver_patients saved — {len(patients_bronze):,} rows")

# Conditions, medications, procedures have no direct PHI
# Just copy them to Silver as-is
for table in ["conditions", "medications", "procedures"]:
    df = spark.table(f"bronze_{table}")
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"silver_{table}")
    print(f"✅ silver_{table} saved")

print("\n🎉 Silver layer complete — all PHI removed!")

Loading spaCy language model...
✅ spaCy model ready!
✅ Presidio engines initialized!

De-identifying patients table...
  ✅ FIRST de-identified
  ✅ LAST de-identified
  ✅ SSN de-identified
  ✅ ADDRESS de-identified
  ✅ CITY de-identified
  ✅ BIRTHDATE de-identified

✅ silver_patients saved — 12,352 rows
✅ silver_conditions saved
✅ silver_medications saved
✅ silver_procedures saved

🎉 Silver layer complete — all PHI removed!


In [0]:
# Verify — compare Bronze vs Silver side by side
print("BRONZE (raw PHI visible):")
spark.table("bronze_patients") \
    .select("FIRST", "LAST", "SSN", "CITY") \
    .show(3, truncate=False)

print("SILVER (PHI removed):")
spark.table("silver_patients") \
    .select("FIRST", "LAST", "SSN", "CITY") \
    .show(3, truncate=False)

BRONZE (raw PHI visible):
+---------+--------------+-----------+-------------+
|FIRST    |LAST          |SSN        |CITY         |
+---------+--------------+-----------+-------------+
|Carson894|O'Hara248     |999-36-1729|Middleborough|
|Lorelei90|Greenfelder433|999-89-4066|Freetown     |
|Heath320 |Mante251      |999-91-9641|Braintree    |
+---------+--------------+-----------+-------------+
only showing top 3 rows
SILVER (PHI removed):
+---------------+----------+--------+----------+
|FIRST          |LAST      |SSN     |CITY      |
+---------------+----------+--------+----------+
|Jeri234        |Crooks415 |<US_SSN>|<PERSON>  |
|María Soledad68|Cortez851 |<US_SSN>|<LOCATION>|
|Bill567        |Gutmann970|<US_SSN>|<LOCATION>|
+---------------+----------+--------+----------+
only showing top 3 rows


In [0]:
# Fix Silver patients — fully replace name columns with <PERSON> tag
from pyspark.sql.functions import lit, col

silver_fixed = spark.table("silver_patients") \
    .withColumn("FIRST", lit("<PERSON>")) \
    .withColumn("LAST", lit("<PERSON>"))

# Overwrite silver_patients
silver_fixed.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_patients")

# Verify
print("SILVER (fully de-identified):")
spark.table("silver_patients") \
    .select("FIRST", "LAST", "SSN", "CITY") \
    .show(3, truncate=False)

SILVER (fully de-identified):
+--------+--------+--------+----------+
|FIRST   |LAST    |SSN     |CITY      |
+--------+--------+--------+----------+
|<PERSON>|<PERSON>|<US_SSN>|<LOCATION>|
|<PERSON>|<PERSON>|<US_SSN>|<LOCATION>|
|<PERSON>|<PERSON>|<US_SSN>|<LOCATION>|
+--------+--------+--------+----------+
only showing top 3 rows


In [0]:
# See the Id column in Silver
spark.table("silver_patients") \
    .select("Id", "FIRST", "LAST", "SSN") \
    .show(3, truncate=False)

+------------------------------------+--------+--------+--------+
|Id                                  |FIRST   |LAST    |SSN     |
+------------------------------------+--------+--------+--------+
|e01f1215-ad82-4281-b8ec-b7c9beac4536|<PERSON>|<PERSON>|<US_SSN>|
|8dfe1ae6-88b6-46a6-bf1c-3013f7e44c0d|<PERSON>|<PERSON>|<US_SSN>|
|d9ea1abc-9b19-4c5f-8917-83b3628d94e0|<PERSON>|<PERSON>|<US_SSN>|
+------------------------------------+--------+--------+--------+
only showing top 3 rows


In [0]:
# Gold layer — join all Silver tables into one agent-ready table
from pyspark.sql.functions import collect_list, concat_ws, col

# Read silver tables
patients = spark.table("silver_patients")
conditions = spark.table("silver_conditions")
medications = spark.table("silver_medications")
procedures = spark.table("silver_procedures")

# Aggregate conditions per patient
conditions_agg = conditions.groupBy("PATIENT") \
    .agg(concat_ws(", ", collect_list("DESCRIPTION")).alias("conditions"))

# Aggregate medications per patient
medications_agg = medications.groupBy("PATIENT") \
    .agg(concat_ws(", ", collect_list("DESCRIPTION")).alias("medications"))

# Aggregate procedures per patient
procedures_agg = procedures.groupBy("PATIENT") \
    .agg(concat_ws(", ", collect_list("DESCRIPTION")).alias("procedures"))

# Join everything together
gold = patients \
    .join(conditions_agg, patients.Id == conditions_agg.PATIENT, "left") \
    .join(medications_agg, patients.Id == medications_agg.PATIENT, "left") \
    .join(procedures_agg, patients.Id == procedures_agg.PATIENT, "left") \
    .select(
        col("Id").alias("patient_id"),
        col("GENDER").alias("gender"),
        col("RACE").alias("race"),
        col("conditions"),
        col("medications"),
        col("procedures")
    )

# Save as Gold Delta table
gold.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_patient_summary")

print(f"✅ gold_patient_summary saved — {gold.count():,} rows")
print("\nSample Gold record:")
gold.show(2, truncate=50)

✅ gold_patient_summary saved — 12,352 rows

Sample Gold record:
+------------------------------------+------+-----+--------------------------------------------------+--------------------------------------------------+----------------------------------------+
|                          patient_id|gender| race|                                        conditions|                                       medications|                              procedures|
+------------------------------------+------+-----+--------------------------------------------------+--------------------------------------------------+----------------------------------------+
|e01f1215-ad82-4281-b8ec-b7c9beac4536|     M|black|Body mass index 30+ - obesity (finding), Corona...|Clopidogrel 75 MG Oral Tablet, Simvastatin 20 M...|   Medication Reconciliation (procedure)|
|8dfe1ae6-88b6-46a6-bf1c-3013f7e44c0d|     F|white|Chronic sinusitis (disorder), Body mass index 3...|24 HR Metformin hydrochloride 500 MG Extended R...|Fac